# 05 Dostrajanie modelu NLI (H4a)

## 1. Konfiguracja

In [1]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/Licencjat/extra

import sys
sys.path.append("..")

import gc

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from src.config import (FINETUNE_BATCH_SIZE, FINETUNE_EPOCHS, FINETUNE_FOLDS, FINETUNE_LEARNING_RATE,
                        FINETUNE_WARMUP_RATIO, FINETUNED_MODEL_DIR, LABELS_DIR, NLI_HYPOTHESIS, NLI_MODEL,
                        PROCESSED_DIR, RANDOM_STATE, RESULTS_DIR)
from src.nli import nli_probabilities

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Brak GPU: Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU"
torch.cuda.get_device_name(0)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Licencjat/extra


'Tesla T4'

## 2. Etykiety i teksty (`labels/stance_train.csv`, `labels/stance_test.csv`: `id`, `term`, `label`)

In [2]:
mentions = pd.read_parquet(PROCESSED_DIR / "h4a" / "mentions.parquet", columns=["id", "term", "text", "target", "cycle"])
labelled = {}
for name in ("train", "test"):
    labels = pd.read_csv(LABELS_DIR / f"stance_{name}.csv", dtype={"id": str})
    labelled[name] = labels.merge(mentions, on=["id", "term"])
    print(name, "etykiet:", len(labels), "| znalezionych tekstów:", len(labelled[name]))
train, test = labelled["train"], labelled["test"]
pd.crosstab(train["target"], train["label"])

train etykiet: 480 | znalezionych tekstów: 480
test etykiet: 200 | znalezionych tekstów: 200


label,0,1
target,,
out,90,150
own,180,60


## 3. Trening i ocena

In [3]:
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
MICRO_BATCH_SIZE = 2


def train_model(texts, terms, labels):
    gc.collect()
    torch.cuda.empty_cache()
    torch.manual_seed(RANDOM_STATE)
    # checkpoint jest zapisany w fp16; trening w fp16 daje NaN, więc wymuszamy fp32
    model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL, dtype=torch.float32).to(device)
    label_ids = {name.lower(): index for name, index in model.config.label2id.items()}
    # etykieta 1 (krytyka) uczy klasy entailment, etykieta 0 – klasy contradiction
    class_of = {1: label_ids["entailment"], 0: label_ids["contradiction"]}
    hypotheses = [NLI_HYPOTHESIS.format(term) for term in terms]

    accumulation_steps = max(1, FINETUNE_BATCH_SIZE // MICRO_BATCH_SIZE)
    steps = FINETUNE_EPOCHS * int(np.ceil(len(texts) / FINETUNE_BATCH_SIZE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LEARNING_RATE, foreach=False)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(FINETUNE_WARMUP_RATIO * steps), steps)

    rng = np.random.default_rng(RANDOM_STATE)
    model.train()
    for epoch in range(FINETUNE_EPOCHS):
        order = rng.permutation(len(texts))
        optimizer.zero_grad()
        epoch_losses = []
        for step, start in enumerate(range(0, len(order), MICRO_BATCH_SIZE)):
            index = order[start:start + MICRO_BATCH_SIZE]
            batch = tokenizer([texts[i] for i in index], [hypotheses[i] for i in index], truncation="only_first",
                              max_length=512, padding=True, return_tensors="pt").to(device)
            targets = torch.tensor([class_of[labels[i]] for i in index], device=device)

            loss = torch.nn.functional.cross_entropy(model(**batch).logits, targets)
            assert torch.isfinite(loss), f"strata = {loss.item()}, epoka {epoch + 1}, krok {step}"
            (loss / accumulation_steps).backward()
            epoch_losses.append(loss.item())

            if (step + 1) % accumulation_steps == 0 or start + MICRO_BATCH_SIZE >= len(order):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
        print(f"epoka {epoch + 1}: średnia strata {np.mean(epoch_losses):.3f}")
    return model.eval()


def score(data):
    sensitivity = data.loc[data["label"] == 1, "predicted"].mean()
    false_alarm = data.loc[data["label"] == 0, "predicted"].mean()
    return {"n": len(data), "kappa": cohen_kappa_score(data["label"], data["predicted"]),
            "f1": f1_score(data["label"], data["predicted"]), "share_predicted": data["predicted"].mean(),
            "sensitivity": sensitivity, "false_alarm": false_alarm,
            "sensitivity_minus_false_alarm": sensitivity - false_alarm}


def score_table(data, by):
    rows = [{"group": "all", **score(data)}]
    for column in by:
        for value, group in data.groupby(column):
            rows.append({"group": f"{column}={value}", **score(group)})
    return pd.DataFrame(rows)

#3.5 Punkt odniesienia

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL, dtype=torch.float32).to(device).eval()
baseline = nli_probabilities(train["text"].tolist(), train["term"].tolist(), model, tokenizer, device, FINETUNE_BATCH_SIZE)
assert baseline["p_entail"].notna().all()
train["predicted"] = baseline["critical"].astype(int).values
baseline_scores = score_table(train, ["target"])
baseline_scores.to_csv(RESULTS_DIR / "validation_stance_baseline.csv", index=False)
del model
gc.collect()
torch.cuda.empty_cache()
baseline_scores.round(3)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

,group,n,kappa,f1,share_predicted,sensitivity,false_alarm,sensitivity_minus_false_alarm
0,all,480,0.319,0.693,0.735,0.929,0.585,0.343
1,target=out,240,0.200,0.777,0.854,0.920,0.744,0.176
2,target=own,240,0.299,0.548,0.617,0.950,0.506,0.444


## 4. Walidacja krzyżowa na zbiorze treningowym

In [5]:
strata = train["label"].astype(str) + "_" + train["target"]
folds = StratifiedKFold(n_splits=FINETUNE_FOLDS, shuffle=True, random_state=RANDOM_STATE)
train["predicted"] = -1
train["p_entail"] = np.nan
fold_scores = []
for fold, (fit_index, held_index) in enumerate(folds.split(train, strata), start=1):
    print(f"fold {fold}")
    used = torch.cuda.memory_allocated() / 1e9
    print(f"pamięć GPU przed treningiem: {used:.1f} GB")
    assert used < 1, "w pamięci GPU został poprzedni model – zrestartuj sesję"
    fit, held = train.loc[fit_index], train.loc[held_index]
    model = train_model(fit["text"].tolist(), fit["term"].tolist(), fit["label"].tolist())
    probabilities = nli_probabilities(held["text"].tolist(), held["term"].tolist(), model, tokenizer, device, FINETUNE_BATCH_SIZE)
    assert probabilities["p_entail"].notna().all(), f"NaN w przewidywaniach, fold {fold}"
    train.loc[held_index, "predicted"] = probabilities["critical"].astype(int).values
    train.loc[held_index, "p_entail"] = probabilities["p_entail"].values
    fold_scores.append({"fold": fold, **score(train.loc[held_index])})
    print(fold_scores[-1])
    del model
    gc.collect()
    torch.cuda.empty_cache()
assert (train["predicted"] != -1).all()
fold_scores = pd.DataFrame(fold_scores)
fold_scores.to_csv(RESULTS_DIR / "validation_stance_cv_folds.csv", index=False)
fold_scores.round(3)

fold 1
pamięć GPU przed treningiem: 0.0 GB


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

epoka 1: średnia strata 0.832
epoka 2: średnia strata 0.247
epoka 3: średnia strata 0.082
{'fold': 1, 'n': 96, 'kappa': np.float64(0.5833333333333333), 'f1': 0.7777777777777778, 'share_predicted': np.float64(0.5), 'sensitivity': np.float64(0.8333333333333334), 'false_alarm': np.float64(0.24074074074074073), 'sensitivity_minus_false_alarm': np.float64(0.5925925925925927)}
fold 2
pamięć GPU przed treningiem: 0.0 GB


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

epoka 1: średnia strata 0.774
epoka 2: średnia strata 0.234
epoka 3: średnia strata 0.125
{'fold': 2, 'n': 96, 'kappa': np.float64(0.6683937823834196), 'f1': 0.8260869565217391, 'share_predicted': np.float64(0.5208333333333334), 'sensitivity': np.float64(0.9047619047619048), 'false_alarm': np.float64(0.2222222222222222), 'sensitivity_minus_false_alarm': np.float64(0.6825396825396826)}
fold 3
pamięć GPU przed treningiem: 0.0 GB


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

... (2 outputs left out)

epoka 1: średnia strata 0.925
epoka 2: średnia strata 0.274
epoka 3: średnia strata 0.080
{'fold': 4, 'n': 96, 'kappa': np.float64(0.6031331592689295), 'f1': 0.7865168539325843, 'share_predicted': np.float64(0.4895833333333333), 'sensitivity': np.float64(0.8333333333333334), 'false_alarm': np.float64(0.2222222222222222), 'sensitivity_minus_false_alarm': np.float64(0.6111111111111112)}
fold 5
pamięć GPU przed treningiem: 0.0 GB


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

epoka 1: średnia strata 0.829
epoka 2: średnia strata 0.269
epoka 3: średnia strata 0.064
{'fold': 5, 'n': 96, 'kappa': np.float64(0.6956521739130435), 'f1': 0.8108108108108109, 'share_predicted': np.float64(0.3333333333333333), 'sensitivity': np.float64(0.7142857142857143), 'false_alarm': np.float64(0.037037037037037035), 'sensitivity_minus_false_alarm': np.float64(0.6772486772486772)}


,fold,n,kappa,f1,share_predicted,sensitivity,false_alarm,sensitivity_minus_false_alarm
0,1,96,0.583,0.778,0.500,0.833,0.241,0.593
1,2,96,0.668,0.826,0.521,0.905,0.222,0.683
2,3,96,0.559,0.759,0.469,0.786,0.222,0.563
3,4,96,0.603,0.787,0.490,0.833,0.222,0.611
4,5,96,0.696,0.811,0.333,0.714,0.037,0.677


In [6]:
cv_scores = score_table(train, ["target"])
cv_scores.to_csv(RESULTS_DIR / "validation_stance_cv.csv", index=False)
cv_scores.round(3)

,group,n,kappa,f1,share_predicted,sensitivity,false_alarm,sensitivity_minus_false_alarm
0,all,480,0.621,0.792,0.462,0.814,0.189,0.625
1,target=out,240,0.516,0.814,0.604,0.800,0.278,0.522
2,target=own,240,0.645,0.745,0.321,0.850,0.144,0.706


In [7]:
# Sprawdźmy rozkład surowych prawdopodobieństw, aby dowiedzieć się, dlaczego klasyfikator binarizuje wszystko jako 0
if 'probabilities' in globals():
    print("Statystyki surowych prawdopodobieństw p_entail:")
    display(probabilities.describe())
else:
    print("Brak zmiennej 'probabilities' w pamięci. Uruchom ponownie walidację krzyżową (krok 4).")

Statystyki surowych prawdopodobieństw p_entail:


,p_entail,p_entail_3
count,96.000000,96.000000
mean,0.344170,0.344074
std,0.467878,0.467761
min,0.000167,0.000167
25%,0.000553,0.000553
50%,0.002238,0.002238
75%,0.999691,0.999422
max,0.999910,0.999794


In [4]:
LEARNING_CURVE_FRACTIONS = [0.5, 0.75, 1.0]

train["stratum"] = train["label"].astype(str) + "_" + train["target"]
folds = StratifiedKFold(n_splits=FINETUNE_FOLDS, shuffle=True, random_state=RANDOM_STATE)
curve_rows = []
for fraction in LEARNING_CURVE_FRACTIONS:
    predicted = pd.Series(-1, index=train.index)
    for fold, (fit_index, held_index) in enumerate(folds.split(train, train["stratum"]), start=1):
        print(f"część {fraction}, fold {fold}")
        assert torch.cuda.memory_allocated() / 1e9 < 1, "w pamięci GPU został poprzedni model – zrestartuj sesję"
        fit = train.loc[fit_index].groupby("stratum").sample(frac=fraction, random_state=RANDOM_STATE)
        held = train.loc[held_index]
        model = train_model(fit["text"].tolist(), fit["term"].tolist(), fit["label"].tolist())
        probabilities = nli_probabilities(held["text"].tolist(), held["term"].tolist(), model, tokenizer, device, FINETUNE_BATCH_SIZE)
        assert probabilities["p_entail"].notna().all(), f"NaN w przewidywaniach, część {fraction}, fold {fold}"
        predicted.loc[held_index] = probabilities["critical"].astype(int).values
        del model
        gc.collect()
        torch.cuda.empty_cache()
    assert (predicted != -1).all()
    curve_rows.append({"fraction": fraction, "n_fit": len(fit), **score(train.assign(predicted=predicted))})
    print(curve_rows[-1])

learning_curve = pd.DataFrame(curve_rows)
learning_curve.to_csv(RESULTS_DIR / "validation_stance_learning_curve.csv", index=False)
learning_curve[["fraction", "n_fit", "kappa", "f1", "share_predicted", "sensitivity_minus_false_alarm"]].round(3)

część 0.5, fold 1


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

epoka 1: średnia strata 1.060
epoka 2: średnia strata 0.342
epoka 3: średnia strata 0.111
część 0.5, fold 2


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

epoka 1: średnia strata 1.082
epoka 2: średnia strata 0.364
epoka 3: średnia strata 0.143
część 0.5, fold 3


... (23 outputs left out)

epoka 1: średnia strata 0.788
epoka 2: średnia strata 0.274
epoka 3: średnia strata 0.084
część 1.0, fold 5


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

epoka 1: średnia strata 0.814
epoka 2: średnia strata 0.291
epoka 3: średnia strata 0.123
{'fraction': 1.0, 'n_fit': 384, 'n': 480, 'kappa': np.float64(0.6134453781512605), 'f1': 0.7880184331797235, 'share_predicted': np.float64(0.4666666666666667), 'sensitivity': np.float64(0.8142857142857143), 'false_alarm': np.float64(0.1962962962962963), 'sensitivity_minus_false_alarm': np.float64(0.617989417989418)}


,fraction,n_fit,kappa,f1,share_predicted,sensitivity_minus_false_alarm
0,0.50,192,0.585,0.767,0.438,0.585
1,0.75,288,0.612,0.785,0.454,0.615
2,1.00,384,0.613,0.788,0.467,0.618


In [8]:
%%writefile ../src/nli.py
import pandas as pd
import torch
from src.config import NLI_HYPOTHESIS, NLI_THRESHOLD

def nli_probabilities(texts, terms, model, tokenizer, device, batch_size=16):
    """p_entail (softmax z entailment i contradiction) oraz p_entail_3 (softmax z trzech klas) dla par tekst–hipoteza."""
    labels = {label.lower(): index for index, label in model.config.id2label.items()}
    entailment, contradiction = labels["entailment"], labels["contradiction"]
    hypotheses = [NLI_HYPOTHESIS.format(term) for term in terms]
    rows = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], hypotheses[start:start + batch_size], truncation="only_first",
                          max_length=512, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**batch).logits.float().cpu()
        p_entail = torch.softmax(logits[:, [entailment, contradiction]], dim=1)[:, 0]
        p_entail_3 = torch.softmax(logits, dim=1)[:, entailment]
        rows.append(pd.DataFrame({"p_entail": p_entail.numpy(), "p_entail_3": p_entail_3.numpy()}))
    result = pd.concat(rows, ignore_index=True)
    result["critical"] = result["p_entail"] >= NLI_THRESHOLD
    result["critical_3"] = result["p_entail_3"] >= NLI_THRESHOLD
    return result

Overwriting ../src/nli.py


## 5. Trening końcowy na całym zbiorze treningowym

In [4]:
model = train_model(train["text"].tolist(), train["term"].tolist(), train["label"].tolist())
FINETUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

epoka 1: średnia strata 0.754
epoka 2: średnia strata 0.270
epoka 3: średnia strata 0.100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Licencjat/processed/h4a/nli_finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/Licencjat/processed/h4a/nli_finetuned/tokenizer.json')

## 6. Ocena na zbiorze testowym

In [5]:
probabilities = nli_probabilities(test["text"].tolist(), test["term"].tolist(), model, tokenizer, device, FINETUNE_BATCH_SIZE)
assert probabilities["p_entail"].notna().all()
test["p_entail"] = probabilities["p_entail"].values
test["predicted"] = probabilities["critical"].astype(int).values
test[["id", "term", "label", "target", "cycle", "p_entail", "predicted"]].to_csv(
    RESULTS_DIR / "validation_stance_test_predictions.csv", index=False)
test_scores = score_table(test, ["target", "cycle"])
test_scores.to_csv(RESULTS_DIR / "validation_stance_test.csv", index=False)
test_scores.round(3)

,group,n,kappa,f1,share_predicted,sensitivity,false_alarm,sensitivity_minus_false_alarm
0,all,200,0.588,0.759,0.415,0.759,0.171,0.588
1,target=out,100,0.454,0.787,0.590,0.762,0.297,0.465
2,target=own,100,0.593,0.682,0.240,0.750,0.112,0.638
3,cycle=2016,100,0.575,0.764,0.450,0.773,0.196,0.576
4,cycle=2024,100,0.599,0.753,0.380,0.744,0.148,0.596
